# KneeVision++ — Clinical Text Training (OAI)

Trains the BioClinicalBERT text model on the real OAI dataset built by `scripts/download_oai.py`.

Run cells top-to-bottom. Set `DATA_CSV` below if your dataset lives elsewhere.

```bash
uv run --extra dev python -m notebooks.to_py train_clinical.ipynb   # or run via: uv run --extra dev python -m jupyter nbclassic --port 8888
```

Local shortcut to launch a kernel:
```bash
uv run --extra dev python -m ipykernel_launcher
```

In [ ]:
# --- Colab Setup --- (optional; runs only on Colab)
import sys, os
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    project_root = Path("/content/drive/MyDrive/KneeVision")
    !pip install torch transformers datasets -q
else:
    project_root = Path.cwd().parent

sys.path.insert(0, str(project_root / "src"))
os.chdir(project_root)
print(f"Project root: {project_root}")
print(f"Working dir:  {os.getcwd()}")

In [ ]:
import torch
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, cohen_kappa_score
from tqdm.auto import tqdm

from kneevision.config.settings import (
    OAI_DATA_DIR, CLINICAL_MAX_LENGTH, CLINICAL_BATCH_SIZE,
    CLINICAL_LEARNING_RATE, MODELS_DIR,
)
from kneevision.clinical.model import ClinicalTextModel
from kneevision.clinical.dataset import ClinicalTextDataset
from kneevision.clinical.prepare import load_reports_csv
from kneevision.data.prepare import class_weights
from kneevision.training.losses import FocalLoss
from kneevision.utils.helpers import set_seed, get_device

DATA_CSV = OAI_DATA_DIR / "processed" / "oai_clinical.csv"
EPOCHS = 10
BATCH_SIZE = CLINICAL_BATCH_SIZE
LR = CLINICAL_LEARNING_RATE
NUM_CLASSES = 5

set_seed(42)
device = get_device()
print(f"Device: {device}")
print(f"Data : {DATA_CSV} (exists: {DATA_CSV.exists()})")

In [ ]:
splits = load_reports_csv(DATA_CSV)

train_texts, train_labels = splits["train"]
val_texts,   val_labels   = splits["val"]
test_texts,  test_labels  = splits["test"]

print(f"Train: {len(train_texts)} | Val: {len(val_texts)} | Test: {len(test_texts)}")
print("\nExample report:")
print(train_texts[0])

In [ ]:
def make_loader(texts, labels, shuffle):
    ds = ClinicalTextDataset(texts, labels, max_length=CLINICAL_MAX_LENGTH)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle, num_workers=0)

train_loader = make_loader(train_texts, train_labels, shuffle=True)
val_loader   = make_loader(val_texts,   val_labels,   shuffle=False)
test_loader  = make_loader(test_texts,  test_labels,  shuffle=False)

In [ ]:
model = ClinicalTextModel(num_classes=NUM_CLASSES).to(device)  # add freeze_encoder=True for a quick smoke run

alpha = torch.tensor(class_weights(train_labels, num_classes=NUM_CLASSES), dtype=torch.float).to(device)
criterion = FocalLoss(alpha=alpha, gamma=2.0, label_smoothing=0.1)

optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

MODELS_DIR.mkdir(exist_ok=True)
print(f"Class weights: {alpha.cpu().tolist()}")
print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

In [ ]:
def validate(model, loader, criterion):
    """Return (avg_loss, accuracy, kappa, preds, labels)."""
    model.eval()
    total_loss = 0.0
    all_preds, all_labels = [], []
    with torch.inference_mode():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            logits = model(input_ids, attention_mask)
            total_loss += criterion(logits, labels).item()
            all_preds.extend(logits.argmax(dim=1).cpu().tolist())
            all_labels.extend(labels.cpu().tolist())
    acc = sum(p == t for p, t in zip(all_preds, all_labels)) / len(all_labels)
    kappa = cohen_kappa_score(all_labels, all_preds, weights="quadratic")
    return total_loss / len(loader), acc, kappa, all_preds, all_labels

In [ ]:
best_kappa = -1.0
train_losses, val_losses, val_accs, val_kappas = [], [], [], []

for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_loss = 0.0
    for batch in tqdm(train_loader, desc=f"Epoch {epoch}", leave=False):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        optimizer.zero_grad(set_to_none=True)
        loss = criterion(model(input_ids, attention_mask), labels)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    scheduler.step()

    val_loss, val_acc, val_kappa, *_ = validate(model, val_loader, criterion)
    train_losses.append(epoch_loss / len(train_loader))
    val_losses.append(val_loss)
    val_accs.append(val_acc)
    val_kappas.append(val_kappa)

    print(f"Epoch {epoch:2d} | Train Loss: {epoch_loss/len(train_loader):.4f} | "
          f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | Val Kappa: {val_kappa:.4f}")

    if val_kappa > best_kappa:
        best_kappa = val_kappa
        torch.save(model.state_dict(), MODELS_DIR / "best_clinical.pt")
        print(f"  -> Saved best model (kappa={val_kappa:.4f})")

print(f"\nBest val kappa: {best_kappa:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(train_losses, label="Train Loss")
axes[0].plot(val_losses, label="Val Loss")
axes[0].set_title("Loss"); axes[0].legend(); axes[0].grid(True)

axes[1].plot(val_accs, color="green", label="Val Accuracy")
axes[1].set_title("Accuracy"); axes[1].legend(); axes[1].grid(True)

axes[2].plot(val_kappas, color="purple", label="Val Kappa")
axes[2].set_title("Quadratic Kappa"); axes[2].legend(); axes[2].grid(True)

for ax in axes:
    ax.set_xlabel("Epoch")
plt.tight_layout()
plt.show()

In [ ]:
model.load_state_dict(torch.load(MODELS_DIR / "best_clinical.pt", map_location=device))
test_loss, test_acc, test_kappa, test_preds, test_labels = validate(model, test_loader, criterion)

print(f"TEST REPORT")
print(f"Test Accuracy: {test_acc:.4f} | Test Kappa: {test_kappa:.4f}")
print(classification_report(test_labels, test_preds, zero_division=0))